# 异步调用
- ainvoke(): 单次调用
- astream()：流式调用
- abatch()： 批量调用

In [1]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
from pathlib import Path
from rich import print as rprint
import asyncio
import time

# 从.env文件中加载环境变量
load_dotenv(Path('../.env'), override=True)
DASHSCOPE_MODEL_NAME = os.getenv('DASHSCOPE_MODEL_NAME')
DASHSCOPE_MODEL_PROVIDER = os.getenv("DASHSCOPE_MODEL_PROVIDER")
DASHSCOPE_API_KEY = os.getenv("DASHSCOPE_API_KEY")
DASHSCOPE_BASE_URL = os.getenv("DASHSCOPE_BASE_URL")

model = init_chat_model(
    model=DASHSCOPE_MODEL_NAME,
    model_provider=DASHSCOPE_MODEL_PROVIDER,
    api_key=DASHSCOPE_API_KEY,
    base_url=DASHSCOPE_BASE_URL
)

## ainvoke

In [8]:
async def demo_async_invoke():
    print("=== 演示：ainvoke 的异步（非阻塞）效果 ===")
    start_time = time.perf_counter()  # 记录开始时间

    print("程序开始...")

    # 1. 创建任务 (Task)
    print(">>> 发起异步模型调用 (ainvoke)...")
    ainvoke_resp = model.ainvoke("用一句话解释人工智能。")
    print(f'ainvoke_resp>>>:{ainvoke_resp}')
    async_task = asyncio.create_task(ainvoke_resp)

    # 2. 并行执行其他任务
    print(">>> 模型请求已在后台发送，继续执行本地逻辑...")
    for i in range(3):
        await asyncio.sleep(1)  # 使用异步等待，释放控制权
        print(f">>> 正在执行第{i + 1}个任务... (已耗时 {time.perf_counter() - start_time:.2f}s)")

    # 3. 获取模型结果
    print(">>> 本地任务完成，检查模型状态...")
    response = await async_task

    end_time = time.perf_counter()
    print(f">>> 模型返回: {response.content}")
    print(f"=== 总运行耗时: {end_time - start_time:.2f}s ===")


async def main():
    """主函数"""
    await demo_async_invoke()


if __name__ == "__main__":
    print('main...')
    # jupyter中不需要这样
    # asyncio.run(main())
    await main()

main...
=== 演示：ainvoke 的异步（非阻塞）效果 ===
程序开始...
>>> 发起异步模型调用 (ainvoke)...
ainvoke_resp>>>:<coroutine object BaseChatModel.ainvoke at 0x000001C2D162F940>
>>> 模型请求已在后台发送，继续执行本地逻辑...
>>> 正在执行第1个任务... (已耗时 1.01s)
>>> 正在执行第2个任务... (已耗时 2.02s)
>>> 正在执行第3个任务... (已耗时 3.03s)
>>> 本地任务完成，检查模型状态...
>>> 模型返回: 人工智能是模拟、延伸和扩展人的智能的一门技术科学，它使机器能够执行通常需要人类智能才能完成的任务。
=== 总运行耗时: 3.03s ===


## astream

In [11]:
async def demo_async_stream():
    """演示异步调用的非阻塞特性"""
    print("=== 演示：astream 的异步（非阻塞）效果 ===")
    start_time = time.perf_counter()  # 记录开始时间
    print("程序开始...")

    # 1. 发起异步流式请求
    # 注意：此时请求已发出，返回的是一个异步生成器
    print(">>> 发起异步流式调用 (astream)...")
    stream_resp = model.astream("写一个10s的分镜脚本")
    print(f'stream_resp>>>:{stream_resp}')

    # 2. 在等待流式响应的同时，执行其他任务
    print(">>> 流式请求已发送，程序无需等待，继续执行其他异步任务...")
    for i in range(3):
        # 使用 asyncio.sleep 而非 time.sleep
        # 这允许事件循环在等待时去处理上面的 stream_resp 网络 IO
        await asyncio.sleep(1)
        # print(f">>> 正在执行并发任务 {i + 1}... ")
        print(f">>> 正在执行第{i + 1}个任务... (已耗时 {time.perf_counter() - start_time:.2f}s)")

    # 3. 现在开始处理流式结果
    print(">>> 模拟任务已完成，开始读取缓冲区中的流式结果...")
    end_time = time.perf_counter()
    print(">>> 流式输出: ", end="", flush=True)
    async for chunk in stream_resp:
        # LangChain 的消息块通常通过 .content 获取内容
        content = chunk.content if hasattr(chunk, 'content') else str(chunk)
        print(content, end="", flush=True)

    print("\n>>> 流式输出结束\n")
    print(f"=== 总运行耗时: {end_time - start_time:.2f}s ===")


async def main():
    """主函数"""
    await demo_async_stream()


if __name__ == "__main__":
    print('main...')
    # jupyter中不需要这样
    # asyncio.run(main())
    await main()

main...
=== 演示：astream 的异步（非阻塞）效果 ===
程序开始...
>>> 发起异步流式调用 (astream)...
stream_resp>>>:<async_generator object BaseChatModel.astream at 0x000001C2D1618800>
>>> 流式请求已发送，程序无需等待，继续执行其他异步任务...
>>> 正在执行第1个任务... (已耗时 1.01s)
>>> 正在执行第2个任务... (已耗时 2.02s)
>>> 正在执行第3个任务... (已耗时 3.02s)
>>> 模拟任务已完成，开始读取缓冲区中的流式结果...
>>> 流式输出: 当然，下面是一个简短的10秒分镜脚本示例。这个例子假设我们正在为一部关于小猫寻找回家路的温馨动画短片制作开头部分。

### 场景1：宁静的公园
- **镜头**：广角镜头从天空缓缓下降至地面。
- **画面**：清晨的第一缕阳光穿透薄雾，照亮了一座安静美丽的公园。
- **时间**：2秒

### 场景2：小猫出现
- **镜头**：切换到中景，跟随一只小猫咪的脚步。
- **画面**：一只可爱的小猫在草地上慢慢行走着，显得有些迷茫。
- **时间**：3秒

### 场景3：环境变化
- **镜头**：快速切换几个视角，展示周围环境的变化（如风吹动树叶、远处孩子们玩耍的声音）。
- **画面**：通过这些快速

D:\developer-tools\anaconda\envs\langchain1.2\Lib\site-packages\anyio\_backends\_asyncio.py:1326: RuntimeWarning: coroutine 'main' was never awaited
  async def receive(self, max_bytes: int = 65536) -> bytes:


切换的画面来营造一种不安定的感觉，暗示小猫可能迷路了。
- **时间**：2秒

### 场景4：小猫反应
- **镜头**：特写镜头聚焦于小猫的脸部表情。
- **画面**：小猫停下脚步，抬头望向四周，眼睛里充满了疑惑和一丝担忧。
- **时间**：3秒

---

这只是一个非常基础的例子，实际根据项目需求可能会有所不同。希望这能给你提供一些灵感！如果有更具体的要求或想要探索其他类型的场景，请随时告诉我。
>>> 流式输出结束

=== 总运行耗时: 3.02s ===


### abatch

In [12]:
async def demo_async_batch():
    """演示异步批量的非阻塞特性"""
    print("=== 演示：abatch 的异步（非阻塞）效果 ===")
    start_time = time.perf_counter()  # 记录开始时间

    print("程序开始...")

    # 准备批量输入
    questions = ["用一句话说明深度学习与传统机器学习的区别", "中国首都在哪里？"]

    # 1. 发起异步批量请求
    # 关键修改：使用 create_task 让协程立即在后台执行
    print(">>> 发起异步批量调用 (abatch)...")
    batch_task = asyncio.create_task(model.abatch(questions))

    # 2. 在等待批量处理的同时，执行其他任务
    print(">>> 批量任务已在后台运行，主程序继续执行...")
    for i in range(3):
        # 关键修改：使用 asyncio.sleep 允许后台任务获取 CPU 时间片进行网络请求
        await asyncio.sleep(1)
        print(f">>> 正在执行第{i + 1}个任务... (已耗时 {time.perf_counter() - start_time:.2f}s)")

    # 3. 等待批量处理结果
    print(">>> 其他任务已完成，现在获取后台批量任务的结果...")
    # 此时 batch_task 可能已经完成，或者我们在这里等待它完成
    responses = await batch_task

    end_time = time.perf_counter()

    for response in responses:
        content = response.content if hasattr(response, 'content') else str(response)
        print(f">>> 响应内容: {content}")

    print(f"=== 总运行耗时: {end_time - start_time:.2f}s ===")


await demo_async_batch()


=== 演示：abatch 的异步（非阻塞）效果 ===
程序开始...
>>> 发起异步批量调用 (abatch)...
>>> 批量任务已在后台运行，主程序继续执行...
>>> 正在执行第1个任务... (已耗时 1.00s)
>>> 正在执行第2个任务... (已耗时 2.03s)
>>> 正在执行第3个任务... (已耗时 3.03s)
>>> 其他任务已完成，现在获取后台批量任务的结果...
>>> 响应内容: 深度学习通过多层神经网络自动学习特征表示，而传统机器学习通常需要人工设计和选择特征。
>>> 响应内容: 中国的首都是北京。
=== 总运行耗时: 3.03s ===
